# 02 — Supervised Fine-tuning LLM dengan LoRA (TRL + PEFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/10-llm-finetuning/02_sft_lora_qwen_sentimen.ipynb)

Sekarang kita fine-tune LLM sungguhan: **`Qwen2.5-0.5B-Instruct`** (modul 08) dilatih agar mengklasifikasikan sentimen kalimat bahasa Indonesia dan **selalu** menjawab dengan satu kata: `positif`, `negatif`, atau `netral`.

Mengapa tugas ini? Karena hasilnya **terukur** (akurasi, macro-F1, dan persentase jawaban yang formatnya valid), sehingga efek fine-tuning jelas terlihat. Kita juga bisa membandingkannya dengan fine-tuning IndoBERT di modul 06 pada data yang sama.

Isi notebook:
1. Format dataset SFT (*prompt–completion* dalam format chat).
2. Baseline: model **sebelum** fine-tuning.
3. Konfigurasi LoRA dan `SFTTrainer` dari library **TRL**.
4. Training dan kurva loss.
5. Evaluasi **sesudah** fine-tuning.
6. Efek samping: apakah kemampuan umum model menurun?
7. Menyimpan, memuat, dan menggabungkan adapter.
8. QLoRA untuk model yang lebih besar.

**Data:** NusaX-Senti bahasa Indonesia ([IndoNLP/nusax](https://github.com/IndoNLP/nusax), lisensi **CC-BY-SA**): 500 train, 100 validasi, 400 test. **Model:** [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (Apache-2.0).

> Aktifkan GPU (**Runtime → Change runtime type → T4 GPU**). Dengan T4, training ±5 menit.

## 0. Cek GPU dan Instalasi

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, berjalan di CPU (jauh lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

In [ ]:
%pip install -q "transformers>=4.56" "trl>=1.0" "peft>=0.17" datasets accelerate scikit-learn pandas matplotlib

Konfigurasi. `QUICK_RUN=1` (pengujian otomatis di CPU) memakai subset sangat kecil dan beberapa langkah training saja, jadi hasilnya tidak bermakna. Jalankan tanpa `QUICK_RUN` di Colab untuk hasil sebenarnya.

T4 tidak mendukung bf16, jadi kita memakai **fp16** mixed precision di sana, dan bf16 di GPU yang mendukung (misalnya A100).

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from trl import SFTConfig, SFTTrainer

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment/indonesian"
LABELS = ["positif", "negatif", "netral"]
device = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16 = torch.cuda.is_available() and not BF16
torch.manual_seed(42)
print(f"device: {device} | bf16: {BF16} | fp16: {FP16} | QUICK_RUN: {QUICK_RUN}")

## 1. Data dan Format SFT

**Supervised fine-tuning (SFT)** melatih model dengan pasangan *input → output yang diinginkan*. Untuk model chat, datanya ditulis dalam format percakapan. TRL mendukung format **prompt–completion**:

```python
{"prompt":     [{"role": "system", ...}, {"role": "user", "content": "<kalimat>"}],
 "completion": [{"role": "assistant", "content": "positif"}]}
```

Dengan format ini, loss **hanya dihitung pada token completion** (jawaban assistant). Model tidak dilatih untuk meniru system prompt atau kalimat pengguna, hanya untuk menghasilkan jawaban yang benar.

In [ ]:
LABEL_ID = {"positive": "positif", "negative": "negatif", "neutral": "netral"}
SYSTEM = ("Klasifikasikan sentimen kalimat berbahasa Indonesia. "
          "Jawab hanya dengan satu kata: positif, negatif, atau netral.")


def muat(split):
    df = pd.read_csv(f"{NUSAX_URL}/{split}.csv")
    df["label"] = df["label"].map(LABEL_ID)
    return df[["text", "label"]]


train_df, valid_df, test_df = muat("train"), muat("valid"), muat("test")
if QUICK_RUN:
    train_df, valid_df, test_df = train_df.sample(32, random_state=0), valid_df.head(8), test_df.sample(16, random_state=0)


def ke_pesan(teks):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": teks}]


def ke_dataset(df):
    return Dataset.from_list([{"prompt": ke_pesan(r.text), "completion": [{"role": "assistant", "content": r.label}]}
                              for r in df.itertuples()])


train_ds, valid_ds = ke_dataset(train_df), ke_dataset(valid_df)
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")
print("Distribusi label train:", train_df["label"].value_counts().to_dict())
train_ds[0]

Muat tokenizer dan model. Seperti di modul 08, setelan generasi bawaan model diganti dengan setelan netral (greedy) agar evaluasi deterministik.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
eos = model.generation_config.eos_token_id
model.generation_config = GenerationConfig(eos_token_id=eos, pad_token_id=tokenizer.pad_token_id)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} juta parameter")

# Satu contoh train dalam bentuk teks setelah chat template
contoh = train_ds[0]
print(tokenizer.apply_chat_template(contoh["prompt"] + contoh["completion"], tokenize=False))

## 2. Baseline: Sebelum Fine-tuning

Selalu ukur model **sebelum** dilatih. Tanpa baseline, kita tidak tahu apakah fine-tuning benar-benar membantu.

Fungsi `prediksi` membuat jawaban secara *batch*. Untuk generasi batch, padding harus di **kiri**, agar token terakhir setiap baris adalah akhir prompt, tempat jawaban dimulai. Jawaban yang bukan salah satu dari tiga label dihitung **format tidak valid** (dan otomatis salah).

In [ ]:
@torch.no_grad()
def prediksi(model, teks_list, batch_size=16):
    model.eval()
    tokenizer.padding_side = "left"
    hasil = []
    for i in range(0, len(teks_list), batch_size):
        prompts = [tokenizer.apply_chat_template(ke_pesan(t), tokenize=False, add_generation_prompt=True)
                   for t in teks_list[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=5, do_sample=False)
        hasil += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]
    tokenizer.padding_side = "right"
    return hasil


def normalisasi(jawaban):
    kata = jawaban.lower().strip(" .!\n").split()
    return kata[0].strip(".,!") if kata else ""


def evaluasi(jawaban_mentah, label):
    pred = [normalisasi(j) for j in jawaban_mentah]
    return {"akurasi": accuracy_score(label, pred),
            "macro-F1": f1_score(label, pred, labels=LABELS, average="macro", zero_division=0),
            "format valid": float(np.mean([p in LABELS for p in pred]))}, pred


mulai = time.perf_counter()
jawaban_awal = prediksi(model, test_df["text"].tolist())
metrik_awal, pred_awal = evaluasi(jawaban_awal, test_df["label"].tolist())
print(f"Sebelum fine-tuning ({time.perf_counter() - mulai:.0f} detik): { {k: round(v, 3) for k, v in metrik_awal.items()} }")
pd.DataFrame({"teks": test_df["text"].str[:60].values, "label": test_df["label"].values,
              "jawaban model": jawaban_awal}).head(8)

## 3. Konfigurasi LoRA dan SFTTrainer

**`LoraConfig`** (notebook 01):
- `r=16`, `lora_alpha=32`: rank adapter dan skalanya.
- `target_modules="all-linear"`: pasang adapter di semua layer linear (proyeksi attention `q/k/v/o` dan MLP). Ini pilihan yang umumnya lebih baik daripada hanya attention.
- `lora_dropout=0.05`: regularisasi ringan.

**`SFTConfig`** (turunan `TrainingArguments` dari modul 05–06), dengan tambahan khusus SFT seperti `max_length`. Learning rate LoRA (`2e-4`) jauh lebih besar daripada full fine-tuning (`1e-5`–`5e-5`), karena adapter mulai dari nol dan jumlahnya sedikit. Batch efektif = `per_device_train_batch_size × gradient_accumulation_steps` = 16.

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

sft_config = SFTConfig(
    output_dir="hasil_sft",
    num_train_epochs=2,
    max_steps=4 if QUICK_RUN else -1,          # QUICK_RUN: beberapa langkah saja
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0 if QUICK_RUN else 5,
    max_length=256,
    logging_steps=1 if QUICK_RUN else 5,
    eval_strategy="steps",
    eval_steps=2 if QUICK_RUN else 16,
    save_strategy="no",
    bf16=BF16,
    fp16=FP16,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds, eval_dataset=valid_ds,
                     processing_class=tokenizer, peft_config=lora_config)
trainer.model.print_trainable_parameters()

Periksa apa yang sebenarnya dilatih: `labels` bernilai `-100` untuk token yang **tidak** dihitung loss-nya. Hanya token jawaban (dan penanda akhir giliran) yang punya label.

In [ ]:
contoh = trainer.train_dataset[0]
ids, labels = contoh["input_ids"], contoh["labels"]
dilatih = [i for i, lab in zip(ids, labels) if lab != -100]
print(f"{len(ids)} token total, {len(dilatih)} token dihitung loss-nya: {tokenizer.decode(dilatih)!r}")

## 4. Training

In [ ]:
mulai = time.perf_counter()
trainer.train()
print(f"Training selesai dalam {time.perf_counter() - mulai:.0f} detik")

log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(7, 3.5))
if "loss" in log:
    ax.plot(log.dropna(subset=["loss"])["step"], log.dropna(subset=["loss"])["loss"], label="train loss")
if "eval_loss" in log:
    ax.plot(log.dropna(subset=["eval_loss"])["step"], log.dropna(subset=["eval_loss"])["eval_loss"], "o-", label="eval loss")
ax.set_xlabel("langkah")
ax.set_ylabel("loss")
ax.legend()
plt.show()

## 5. Evaluasi Sesudah Fine-tuning

`trainer.model` adalah model dasar yang sudah dipasangi adapter LoRA hasil training.

In [ ]:
model_ft = trainer.model
jawaban_akhir = prediksi(model_ft, test_df["text"].tolist())
metrik_akhir, pred_akhir = evaluasi(jawaban_akhir, test_df["label"].tolist())

pd.DataFrame({"sebelum fine-tuning": metrik_awal, "sesudah LoRA SFT": metrik_akhir}).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, judul in [(axes[0], pred_awal, "Sebelum"), (axes[1], pred_akhir, "Sesudah")]:
    ConfusionMatrixDisplay(confusion_matrix(test_df["label"], pred, labels=LABELS), display_labels=LABELS).plot(
        ax=ax, colorbar=False)
    ax.set_title(f"{judul} fine-tuning")
plt.tight_layout()
plt.show()

Biasanya terlihat dua perbaikan: **format** jawaban menjadi konsisten (hampir 100% valid), dan **akurasi** naik karena model belajar definisi sentimen dari data NusaX.

Bandingkan juga dengan modul 06: IndoBERT (±124 juta parameter, encoder) yang di-fine-tune untuk klasifikasi bisa mencapai hasil serupa atau lebih baik dengan komputasi lebih kecil. Untuk **klasifikasi murni**, encoder kecil sering lebih efisien. SFT pada LLM lebih bernilai saat output berupa teks bebas atau terstruktur (ringkasan, ekstraksi JSON, jawaban dengan gaya tertentu). Klasifikasi dipakai di sini karena mudah diukur.

## 6. Efek Samping: Apakah Kemampuan Umum Menurun?

Fine-tuning pada tugas sempit bisa membuat model "lupa" kemampuan lain (*catastrophic forgetting*, notebook 01). Dua cara cepat memeriksanya:
1. **Perplexity** pada teks umum (bukan data training), dengan dan tanpa adapter. Perplexity = `exp(loss)`, makin rendah makin baik model "memahami" teks tersebut.
2. **Pertanyaan umum** sebelum dan sesudah.

`peft` memudahkan perbandingan ini: `with model_ft.disable_adapter():` menjalankan model dasar tanpa adapter, tanpa perlu memuat model kedua.

In [ ]:
TEKS_UMUM = [
    "Jakarta adalah ibu kota Indonesia dan kota terbesar di negara ini. Kota ini terletak di pesisir utara Pulau Jawa.",
    "Fotosintesis adalah proses tumbuhan mengubah cahaya matahari, air, dan karbon dioksida menjadi glukosa dan oksigen.",
    "Untuk membuat nasi goreng, tumis bawang merah dan bawang putih, lalu masukkan nasi, kecap, dan telur.",
]


@torch.no_grad()
def perplexity(model, teks_list):
    model.eval()
    loss = []
    for t in teks_list:
        enc = tokenizer(t, return_tensors="pt").to(model.device)
        loss.append(model(**enc, labels=enc["input_ids"]).loss.item())   # rata-rata cross-entropy per token
    return float(np.exp(np.mean(loss)))


def jawab_umum(model, pertanyaan, n_token=60):
    enc = tokenizer(tokenizer.apply_chat_template([{"role": "user", "content": pertanyaan}], tokenize=False,
                                                  add_generation_prompt=True), return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=20 if QUICK_RUN else n_token, do_sample=False)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


PERTANYAAN_UMUM = "Sebutkan tiga manfaat olahraga pagi."
with model_ft.disable_adapter():
    ppl_dasar, jawab_dasar = perplexity(model_ft, TEKS_UMUM), jawab_umum(model_ft, PERTANYAAN_UMUM)
ppl_ft, jawab_ft = perplexity(model_ft, TEKS_UMUM), jawab_umum(model_ft, PERTANYAAN_UMUM)

print(f"Perplexity teks umum | tanpa adapter: {ppl_dasar:.2f} | dengan adapter: {ppl_ft:.2f}\n")
print(f"❓ {PERTANYAAN_UMUM}\n--- tanpa adapter ---\n{jawab_dasar}\n--- dengan adapter ---\n{jawab_ft}")

Sering kali model dengan adapter cenderung menjawab singkat, atau bahkan menjawab "positif"/"netral" untuk pertanyaan umum, karena ia hanya pernah dilatih menjawab satu kata. Ini wajar untuk model khusus satu tugas, dan menjadi alasan adapter disimpan **terpisah**: model dasar tetap utuh untuk tugas lain. Jika model perlu tetap serbaguna, campurkan data instruksi umum ke data training.

## 7. Menyimpan, Memuat, dan Menggabungkan Adapter

In [ ]:
from pathlib import Path

folder_adapter = Path("adapter_sentimen")
model_ft.save_pretrained(folder_adapter)
tokenizer.save_pretrained(folder_adapter)

ukuran_adapter = sum(f.stat().st_size for f in folder_adapter.glob("*.safetensors")) / 1e6
ukuran_model = sum(p.numel() for p in model_ft.base_model.parameters()) * 2 / 1e6   # jika disimpan bf16/fp16
print(f"Ukuran adapter: {ukuran_adapter:.1f} MB vs model penuh ±{ukuran_model:.0f} MB (fp16)")
print(sorted(f.name for f in folder_adapter.iterdir()))

Memuat ulang: muat model dasar, lalu pasang adapter dengan `PeftModel.from_pretrained`. Untuk deployment, `merge_and_unload()` menggabungkan adapter ke bobot (seperti `merge()` di notebook 01), sehingga hasilnya model `transformers` biasa tanpa tambahan latensi.

In [ ]:
dasar = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
dasar.generation_config = GenerationConfig(eos_token_id=eos, pad_token_id=tokenizer.pad_token_id)
dimuat = PeftModel.from_pretrained(dasar, folder_adapter)

sampel = test_df["text"].tolist()[:8]
print("Prediksi sama dengan model hasil training:", prediksi(dimuat, sampel) == prediksi(model_ft, sampel))

model_gabung = dimuat.merge_and_unload()
print(type(model_gabung).__name__, "| prediksi setelah merge sama:", prediksi(model_gabung, sampel) == prediksi(model_ft, sampel))

**Membagikan adapter ke Hugging Face Hub.** Simpan token Hugging Face (dengan izin *write*) di Colab Secrets bernama `HF_TOKEN`, **jangan** di kode:

```python
from google.colab import userdata
model_ft.push_to_hub("nama-kamu/qwen2.5-0.5b-sentimen-lora", token=userdata.get("HF_TOKEN"))
```

Cantumkan di model card: model dasar, dataset beserta lisensinya (NusaX: CC-BY-SA), hyperparameter, dan hasil evaluasi.

## 8. QLoRA untuk Model yang Lebih Besar

Model 0,5B muat di T4 tanpa kuantisasi. Untuk model 7B–8B, muat model dasar dalam 4-bit dengan `bitsandbytes` (hanya GPU NVIDIA), lalu latih adapter LoRA seperti biasa. Konfigurasi umumnya:

```python
# %pip install -q bitsandbytes
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # format NormalFloat 4-bit
    bnb_4bit_use_double_quant=True,         # kuantisasi juga konstanta skalanya
    bnb_4bit_compute_dtype=torch.float16,   # T4: float16; A100: torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", quantization_config=bnb_config, device_map="auto")
# lalu SFTTrainer(..., peft_config=lora_config) seperti di atas; aktifkan gradient_checkpointing=True di SFTConfig
```

Kode QLoRA ini tidak dijalankan di notebook (butuh GPU dan unduhan ±15 GB); coba sebagai Latihan 3. Periksa lisensi setiap model sebelum dipakai: tidak semua model terbuka berlisensi Apache-2.0/MIT.

## Ringkasan

- **SFT** melatih pasangan prompt → jawaban; format **prompt–completion** membuat loss hanya dihitung pada jawaban.
- **LoRA dengan `peft` + `SFTTrainer` dari TRL**: hanya ±1–2% parameter yang dilatih, adapter hanya puluhan MB.
- Selalu bandingkan dengan **baseline sebelum fine-tuning**, di set test yang tidak dipakai training, dengan metrik yang relevan (termasuk validitas format).
- Periksa **efek samping** (perplexity & kemampuan umum) dengan `disable_adapter()`.
- Adapter bisa disimpan terpisah, dimuat ulang, atau di-**merge** untuk deployment. **QLoRA** membuka fine-tuning model 7B di GPU 16 GB.

## Latihan

1. **Baseline prompting yang lebih kuat.** Sebelum fine-tuning, coba prompt few-shot (3–6 contoh dari data train di dalam `messages`). Seberapa dekat hasilnya dengan model LoRA? Hitung juga selisih panjang prompt (token) antara few-shot dan model fine-tune. Apa artinya untuk biaya inference?
2. **Hyperparameter.** Latih ulang dengan `r` 4 vs 16 vs 64, dan dengan `target_modules=["q_proj", "v_proj"]` vs `"all-linear"`. Catat akurasi test, jumlah parameter yang dilatih, dan waktu training dalam satu tabel.
3. **Output terstruktur.** Ubah tugasnya: model harus menjawab JSON `{"sentimen": "...", "alasan": "..."}`. Buat data train dengan bantuan Claude (modul 08) untuk kolom `alasan` dari 200 kalimat, lalu latih dan ukur persentase output yang merupakan JSON valid sebelum dan sesudah fine-tuning. (Opsional: coba dengan QLoRA pada model 1,5B–3B.)